In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import geodatasets
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, BoundaryNorm
from matplotlib.pyplot import get_cmap
import colorcet as cc
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# enable latex plotting 
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

import constants as c 
from logger import setup_logger 
log = setup_logger("generate-flood-risk-coverage-maps")
log.setLevel("INFO")
log.info("Modules loaded.")

/share/ju/matt/bayflood/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Modules loaded.


In [2]:
ct_nyc = gpd.read_file(f"{c.GEO_PATH}/ct-nyc-2020.geojson", crs=c.WGS).to_crs(c.PROJ)
log.info("Loaded NYC Census Tracts.")

nybb = gpd.read_file(geodatasets.get_path("nybb"), crs=c.WGS).to_crs(c.PROJ)
log.info("Loaded NYC Boroughs.")

/share/ju/matt/bayflood/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option CRS
  return ogr_read(


2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Loaded NYC Census Tracts.


/share/ju/matt/bayflood/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver ESRI Shapefile does not support open option CRS
  return ogr_read(


2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Loaded NYC Boroughs.


In [3]:
analysis_df = pd.read_csv(c.CURRENT_DF)
log.info("Analysis dataframe loaded.")

# ESTIMATE_THRES should be the 25th quantile of p_y among tracts with confirmed_flooding_image 
analysis_df['confirmed_flooding_image'] = analysis_df['at_least_one_positive_image_by_area'] == 1
log.info(f"Found {analysis_df['confirmed_flooding_image'].sum()} tracts with confirmed flooding images.")

# calculate the 25th quantile of p_y among tracts with confirmed flooding images
ESTIMATE_THRES = analysis_df.loc[analysis_df['confirmed_flooding_image'], 'p_y'].quantile(0.25)
log.info(f"25th quantile of p_y among tracts with confirmed flooding images: {ESTIMATE_THRES}")

2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Analysis dataframe loaded.


2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Found 176 tracts with confirmed flooding images.


2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - 25th quantile of p_y among tracts with confirmed flooding images: 0.006982504058158325


In [4]:
analysis_df['any_sensors'] = analysis_df['n_floodnet_sensors'] > 0
log.info(f"Found {analysis_df['any_sensors'].sum()} tracts with at least one FloodNet sensor.")

# get all columns with 311 in name, and sum them up
analysis_df['n_311_requests'] = analysis_df.filter(like='311').sum(axis=1)
log.info(f"Found {analysis_df['n_311_requests'].sum()} 311 requests.")

analysis_df['any_311_report'] = analysis_df['n_311_requests'] > 0
log.info(f"Found {analysis_df['any_311_report'].sum()} tracts with at least one 311 report.")

analysis_df['no_dep_flooding'] = analysis_df['dep_moderate_2_frac'] == 0
log.info(f"Found {analysis_df['no_dep_flooding'].sum()} tracts with no DEP flooding.")

2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Found 192 tracts with at least one FloodNet sensor.


2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Found 2171 311 requests.


2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Found 878 tracts with at least one 311 report.


2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Found 1001 tracts with no DEP flooding.


In [5]:
# merge geometry from ct_nyc into analysis_df
analysis_df['GEOID'] = analysis_df['GEOID'].astype(str)
analysis_df = ct_nyc.merge(analysis_df, on='GEOID')
# make sure analysis_df has the same number of rows as ct_nyc 
if len(analysis_df) != len(ct_nyc):
    log.error(f"Length of analysis_df ({len(analysis_df)}) does not match length of ct_nyc ({len(ct_nyc)}).")
    exit(1)
else: 
    log.info(f"Length of analysis_df ({len(analysis_df)}) matches length of ct_nyc ({len(ct_nyc)}).")

analysis_df = gpd.GeoDataFrame(analysis_df, crs=c.PROJ)

2026-05-31 12:30:46 - generate-flood-risk-coverage-maps - INFO - Length of analysis_df (2325) matches length of ct_nyc (2325).


In [6]:
from palettable.colorbrewer.sequential import GnBu_5 as colormap

def plot_isolated_signal_map(
    analysis_df,
    estimate='at_least_one_positive_image_by_area',
    figsize=(25, 25),
    title='Tracts with Model Signal Only'
):
    """
    Map showing tracts with signal from estimate column but no other external signals
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Isolate tracts with model signal but no other signals
    isolated_tracts = analysis_df[
        (analysis_df['confirmed_flooding_image'] | (analysis_df[estimate] > ESTIMATE_THRES)) & 
        (analysis_df['any_sensors'] == 0) & 
        (analysis_df['any_311_report'] == 0) & 
        (analysis_df['no_dep_flooding'] == 1)
    ]
    
    # Plot base map in light grey
    analysis_df.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.5, zorder=1)
    
    # Plot isolated tracts in highlight color
    isolated_tracts.plot(ax=ax, color='gold', edgecolor='black', linewidth=1, zorder=2)
    
    # Add nybb boundary
    nybb.boundary.plot(ax=ax, color='black', linewidth=2, zorder=3)
    
    # Add legend
    legend_elements = [
        Patch(facecolor='gold', edgecolor='black', label='Model Signal Only')
    ]
    #ax.legend(handles=legend_elements, loc='upper left', fontsize=24)
    
    #ax.set_title(title, fontsize=28, pad=20)
    ax.axis('off')
    
    return fig, ax

def plot_no_311_map(
    analysis_df,
    estimate='at_least_one_positive_image_by_area',
    figsize=(25, 25),
    title='Tracts with Model Signal but No 311 Reports'
):
    """
    Map showing tracts with signal from estimate column but no 311 reports
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Isolate tracts with model signal but no 311
    target_tracts = analysis_df[
        (analysis_df['confirmed_flooding_image'] | (analysis_df[estimate] > ESTIMATE_THRES)) & 
        (analysis_df['any_311_report'] == 0)
    ]
    
    # Plot base map in light grey
    analysis_df.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.5, zorder=1)
    
    # Plot target tracts in highlight color
    target_tracts.plot(ax=ax, color='purple', edgecolor='black', linewidth=1, zorder=2)
    
    # Add nybb boundary
    nybb.boundary.plot(ax=ax, color='black', linewidth=2, zorder=3)
    
    # Add legend
    legend_elements = [
        Patch(facecolor='purple', edgecolor='black', label='Model Signal without 311 Reports')
    ]
    #ax.legend(handles=legend_elements, loc='upper left', fontsize=24)
    
    #ax.set_title(title, fontsize=28, pad=20)
    ax.axis('off')
    
    return fig, ax

def plot_no_floodnet_map(
    analysis_df,
    estimate='at_least_one_positive_image_by_area',
    figsize=(25, 25),
    title='Tracts with Model Signal but No Floodnet Signal'
):
    """
    Map showing tracts with signal from estimate column but no Floodnet signal
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Isolate tracts with model signal but no floodnet
    target_tracts = analysis_df[
        (analysis_df['confirmed_flooding_image'] | (analysis_df[estimate] > ESTIMATE_THRES)) & 
        (analysis_df['any_sensors'] == 0)
    ]
    
    # Plot base map in light grey
    analysis_df.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.5, zorder=1)
    
    # Plot target tracts in highlight color
    target_tracts.plot(ax=ax, color='darkgreen', edgecolor='black', linewidth=1, zorder=2)
    
    # Add nybb boundary
    nybb.boundary.plot(ax=ax, color='black', linewidth=2, zorder=3)
    
    # Add legend
    legend_elements = [
        Patch(facecolor='darkgreen', edgecolor='black', label='Model Signal without Floodnet')
    ]
    #ax.legend(handles=legend_elements, loc='upper left', fontsize=24)
    
    #ax.set_title(title, fontsize=28, pad=20)
    ax.axis('off')
    
    return fig, ax

def plot_no_dep_map(
    analysis_df,
    estimate='at_least_one_positive_image_by_area',
    figsize=(25, 25),
    title='Tracts with Model Signal but No DEP Predicted Stormwater'
):
    """
    Map showing tracts with signal from estimate column but no DEP stormwater prediction
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Isolate tracts with model signal but no dep prediction
    target_tracts = analysis_df[
        (analysis_df['confirmed_flooding_image'] | (analysis_df[estimate] > ESTIMATE_THRES)) & 
        (analysis_df['no_dep_flooding'] == 1)
    ]
    
    # Plot base map in light grey
    analysis_df.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.5, zorder=1)
    
    # Plot target tracts in highlight color
    target_tracts.plot(ax=ax, color='darkred', edgecolor='black', linewidth=1, zorder=2)
    
    # Add nybb boundary
    nybb.boundary.plot(ax=ax, color='black', linewidth=2, zorder=3)
    
    # Add legend
    legend_elements = [
        Patch(facecolor='darkred', edgecolor='black', label='Model Signal without DEP Prediction')
    ]
    #ax.legend(handles=legend_elements, loc='upper left', fontsize=24)
    
    #ax.set_title(title, fontsize=28, pad=20)
    ax.axis('off')
    
    return fig, ax

# Function to create all four maps
def create_all_maps(analysis_df, estimate='p_y', figsize=(25, 25)):
    """
    Creates all four maps showing different isolation scenarios
    """
    maps = {
        'isolated_signal': plot_isolated_signal_map(analysis_df, estimate, figsize),
        'no_311': plot_no_311_map(analysis_df, estimate, figsize),
        'no_floodnet': plot_no_floodnet_map(analysis_df, estimate, figsize),
        'no_dep': plot_no_dep_map(analysis_df, estimate, figsize)
    }
    return maps

In [7]:
# save each map to paper dir / figures 
for name, (fig, ax) in create_all_maps(analysis_df, estimate='p_y', figsize=(6, 6)).items():
    fig.savefig(f"{c.PAPER_PATH}/figures/{name}.png", dpi=150, bbox_inches='tight', pad_inches=0.025)
    log.info(f"Saved {name} map to ../../paper/figures/map_model_{name}.png.")
    plt.close(fig)

2026-05-31 12:30:50 - generate-flood-risk-coverage-maps - INFO - Saved isolated_signal map to ../../paper/figures/map_model_isolated_signal.png.


2026-05-31 12:30:50 - generate-flood-risk-coverage-maps - INFO - Saved no_311 map to ../../paper/figures/map_model_no_311.png.


2026-05-31 12:30:50 - generate-flood-risk-coverage-maps - INFO - Saved no_floodnet map to ../../paper/figures/map_model_no_floodnet.png.


2026-05-31 12:30:50 - generate-flood-risk-coverage-maps - INFO - Saved no_dep map to ../../paper/figures/map_model_no_dep.png.


In [8]:
# Calculate additional thresholds
ESTIMATE_THRES_10 = analysis_df.loc[analysis_df['confirmed_flooding_image'], 'p_y'].quantile(0.10)
ESTIMATE_THRES_50 = analysis_df.loc[analysis_df['confirmed_flooding_image'], 'p_y'].quantile(0.50)

log.info(f"10th quantile of p_y among tracts with confirmed flooding images: {ESTIMATE_THRES_10}")
log.info(f"50th quantile of p_y among tracts with confirmed flooding images: {ESTIMATE_THRES_50}")

2026-05-31 12:30:50 - generate-flood-risk-coverage-maps - INFO - 10th quantile of p_y among tracts with confirmed flooding images: 0.0040074063026279


2026-05-31 12:30:50 - generate-flood-risk-coverage-maps - INFO - 50th quantile of p_y among tracts with confirmed flooding images: 0.01407932742764925


In [9]:
# Generate and save maps for 10th percentile
original_thres = ESTIMATE_THRES
ESTIMATE_THRES = ESTIMATE_THRES_10

log.info("Generating maps for 10th percentile threshold...")
for name, (fig, ax) in create_all_maps(analysis_df, estimate='p_y', figsize=(6, 6)).items():
    filename = f"{name}_p10"
    fig.savefig(f"{c.PAPER_PATH}/figures/{filename}.png", dpi=150, bbox_inches='tight', pad_inches=0.025)
    log.info(f"Saved {name} map to {c.PAPER_PATH}/figures/{filename}.png")
    plt.close(fig)

# Generate and save maps for 50th percentile
ESTIMATE_THRES = ESTIMATE_THRES_50

log.info("Generating maps for 50th percentile threshold...")
for name, (fig, ax) in create_all_maps(analysis_df, estimate='p_y', figsize=(6, 6)).items():
    filename = f"{name}_p50"
    fig.savefig(f"{c.PAPER_PATH}/figures/{filename}.png", dpi=150, bbox_inches='tight', pad_inches=0.025)
    log.info(f"Saved {name} map to {c.PAPER_PATH}/figures/{filename}.png")
    plt.close(fig)

# Restore original threshold
ESTIMATE_THRES = original_thres

2026-05-31 12:30:50 - generate-flood-risk-coverage-maps - INFO - Generating maps for 10th percentile threshold...


2026-05-31 12:30:53 - generate-flood-risk-coverage-maps - INFO - Saved isolated_signal map to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/isolated_signal_p10.png


2026-05-31 12:30:53 - generate-flood-risk-coverage-maps - INFO - Saved no_311 map to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_311_p10.png


2026-05-31 12:30:53 - generate-flood-risk-coverage-maps - INFO - Saved no_floodnet map to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_floodnet_p10.png


2026-05-31 12:30:53 - generate-flood-risk-coverage-maps - INFO - Saved no_dep map to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_dep_p10.png


2026-05-31 12:30:53 - generate-flood-risk-coverage-maps - INFO - Generating maps for 50th percentile threshold...


2026-05-31 12:30:56 - generate-flood-risk-coverage-maps - INFO - Saved isolated_signal map to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/isolated_signal_p50.png


2026-05-31 12:30:56 - generate-flood-risk-coverage-maps - INFO - Saved no_311 map to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_311_p50.png


2026-05-31 12:30:56 - generate-flood-risk-coverage-maps - INFO - Saved no_floodnet map to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_floodnet_p50.png


2026-05-31 12:30:56 - generate-flood-risk-coverage-maps - INFO - Saved no_dep map to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_dep_p50.png


In [10]:
def plot_joint_threshold_map(
    analysis_df,
    condition_mask,
    t10, t25, t50,
    colors,
    figsize=(25, 25)
):
    fig, ax = plt.subplots(figsize=figsize)
    
    # Base map in light grey
    analysis_df.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.5, zorder=1)
    
    # Identify tracts at each threshold level (highest they reach)
    plot_df = analysis_df.copy()
    plot_df['level'] = 0
    plot_df.loc[condition_mask & (plot_df['confirmed_flooding_image'] | (plot_df['p_y'] > t10)), 'level'] = 1
    plot_df.loc[condition_mask & (plot_df['confirmed_flooding_image'] | (plot_df['p_y'] > t25)), 'level'] = 2
    plot_df.loc[condition_mask & (plot_df['confirmed_flooding_image'] | (plot_df['p_y'] > t50)), 'level'] = 3
    
    # Plotting levels from lightest to darkest
    # Level 1: 10th percentile (Lightest)
    plot_df[plot_df['level'] == 1].plot(ax=ax, color=colors[0], edgecolor='none', zorder=2)
    # Level 2: 25th percentile (Medium)
    plot_df[plot_df['level'] == 2].plot(ax=ax, color=colors[1], edgecolor='none', zorder=3)
    # Level 3: 50th percentile (Darkest)
    plot_df[plot_df['level'] == 3].plot(ax=ax, color=colors[2], edgecolor='none', zorder=4)
    
    # Add thin black outlines for all highlighted tracts to make them pop
    plot_df[plot_df['level'] > 0].plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.25, zorder=5)
    
    # Add nybb boundary
    nybb.boundary.plot(ax=ax, color='black', linewidth=2, zorder=6)
    
    ax.axis('off')
    return fig, ax

# Scenario masks and their specific monochromatic color schemes
# Order: [Lightest (p10), Medium (p25), Darkest (p50)]
scenarios = {
    'isolated_signal': {
        'mask': (analysis_df['any_sensors'] == 0) & (analysis_df['any_311_report'] == 0) & (analysis_df['no_dep_flooding'] == 1),
        'colors': ['#FFF27F', '#FFD700', '#B8860B'] # Shades of Gold
    },
    'no_311': {
        'mask': (analysis_df['any_311_report'] == 0),
        'colors': ['#D8BFD8', '#800080', '#4B0082'] # Shades of Purple
    },
    'no_floodnet': {
        'mask': (analysis_df['any_sensors'] == 0),
        'colors': ['#90EE90', '#006400', '#003300'] # Shades of Dark Green
    },
    'no_dep': {
        'mask': (analysis_df['no_dep_flooding'] == 1),
        'colors': ['#F08080', '#8B0000', '#4D0000'] # Shades of Dark Red
    }
}

log.info("Generating scenario-specific joint threshold maps...")
for name, config in scenarios.items():
    fig, ax = plot_joint_threshold_map(
        analysis_df, 
        config['mask'], 
        ESTIMATE_THRES_10, original_thres, ESTIMATE_THRES_50,
        config['colors'],
        figsize=(6, 6)
    )
    filename = f"{name}_joint"
    fig.savefig(f"{c.PAPER_PATH}/figures/{filename}.png", dpi=150, bbox_inches='tight', pad_inches=0.025)
    log.info(f"Saved {name} joint map with custom colors to {c.PAPER_PATH}/figures/{filename}.png")
    plt.close(fig)

2026-05-31 12:30:56 - generate-flood-risk-coverage-maps - INFO - Generating scenario-specific joint threshold maps...


2026-05-31 12:30:57 - generate-flood-risk-coverage-maps - INFO - Saved isolated_signal joint map with custom colors to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/isolated_signal_joint.png


2026-05-31 12:30:58 - generate-flood-risk-coverage-maps - INFO - Saved no_311 joint map with custom colors to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_311_joint.png


2026-05-31 12:30:59 - generate-flood-risk-coverage-maps - INFO - Saved no_floodnet joint map with custom colors to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_floodnet_joint.png


2026-05-31 12:31:00 - generate-flood-risk-coverage-maps - INFO - Saved no_dep joint map with custom colors to /share/ju/matt/bayflood/papers/natcities_bayflood_2025/figures/no_dep_joint.png
